## 1. Environment Setup / 环境配置

This section checks the Python environment, project path, and available device.

本节检查 Python 环境、项目路径以及可用的训练设备。

In [2]:
import sys
from pathlib import Path

import torch

In [3]:
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)

Python version: 3.11.2 (v3.11.2:878ead1ac1, Feb  7 2023, 10:02:41) [Clang 13.0.0 (clang-1300.0.29.30)]
PyTorch version: 2.12.1


In [5]:
#检查当前项目路径
PROJECT_ROOT = Path.cwd()

print("Project folder:", PROJECT_ROOT)

Project folder: /Users/haruka/Desktop/UNSW/9444/gp


In [6]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Selected device:", DEVICE)

Selected device: mps


## 2. Reproducibility / 实验可复现性

Random seeds are fixed so that dataset splitting, image pairing, and model
initialisation are as reproducible as possible.

固定随机种子，使数据划分、图像配对和模型参数初始化尽可能可复现。

In [7]:
import random

import numpy as np


SEED = 42


def seed_everything(seed: int) -> None:
    """固定 Python、NumPy 和 PyTorch 的随机数生成器。"""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

print("Random seed:", SEED)

Random seed: 42


## 3. Dataset Paths and Image Scanning / 数据路径与图片扫描

In [11]:
DATA_ROOT = PROJECT_ROOT / "data"

COCO_TRAIN_ROOT = DATA_ROOT / "coco" / "train2014"
COCO_VAL_ROOT = DATA_ROOT / "coco" / "val2014"
WIKIART_ROOT = DATA_ROOT / "wikiart"

print("Data root:", DATA_ROOT)
print("COCO train:", COCO_TRAIN_ROOT)
print("COCO validation:", COCO_VAL_ROOT)
print("WikiArt:", WIKIART_ROOT)

print("COCO train exists:", COCO_TRAIN_ROOT.exists())
print("COCO validation exists:", COCO_VAL_ROOT.exists())
print("WikiArt exists:", WIKIART_ROOT.exists())

Data root: /Users/haruka/Desktop/UNSW/9444/gp/data
COCO train: /Users/haruka/Desktop/UNSW/9444/gp/data/coco/train2014
COCO validation: /Users/haruka/Desktop/UNSW/9444/gp/data/coco/val2014
WikiArt: /Users/haruka/Desktop/UNSW/9444/gp/data/wikiart
COCO train exists: True
COCO validation exists: True
WikiArt exists: True


In [19]:
#定义我想要的图片格式
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp",
}

#扫描图片
def list_images(root:Path):
    image_paths = []
    if not root.exists():
        return image_paths
    #rglob("*")：递归查找文件夹中的所有项目
    for path in root.rglob("*"):
        #检查是不是图片文件而不是文件夹，并且把后缀变为小写后在不在图片格式内
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            image_paths.append(path)
    return sorted(image_paths)

#调用上面的函数扫描三个数据文件夹
raw_content_train = list_images(COCO_TRAIN_ROOT)
raw_content_val = list_images(COCO_VAL_ROOT)
raw_style_images = list_images(WIKIART_ROOT)

#print(raw_content_train)
#输出图片数量
print("COCO training images:", len(raw_content_train))
print("COCO held-out images:", len(raw_content_val))
print("WikiArt images:", len(raw_style_images))

#查看路径示例
print("First COCO image:")
print(raw_content_train[0])

print("\nFirst WikiArt image:")
print(raw_style_images[0])

COCO training images: 800
COCO held-out images: 200
WikiArt images: 2000
First COCO image:
/Users/haruka/Desktop/UNSW/9444/gp/data/coco/train2014/COCO_train2014_000000000009.jpg

First WikiArt image:
/Users/haruka/Desktop/UNSW/9444/gp/data/wikiart/Abstract_Expressionism/wikiart_000262.jpg


In [20]:
#检查数据是否准备完成
DATA_READY = (
    len(raw_content_train) > 0 and len(raw_content_val) > 0 and len(raw_style_images) > 0
)

print(DATA_READY)

True


## 4. Train, Validation, and Test Splits / 数据集划分

The content and style images are divided into training, validation, and test sets.

内容图像和风格图像被划分为训练集、验证集和测试集。

- Training set: updates model parameters.
- Validation set: selects checkpoints and hyperparameters.
- Test set: performs final evaluation only.

- 训练集：更新模型参数。
- 验证集：选择 checkpoint 和超参数。
- 测试集：只用于最终评估。

In [22]:
#验证和测试各留下100张
content_train = raw_content_train.copy()
content_held_out = raw_content_val.copy()
#随机打乱留出数据
content_rng = random.Random(SEED)
content_rng.shuffle(content_held_out)

middle_index = len(content_held_out) // 2

content_val = content_held_out[:middle_index]
content_test = content_held_out[middle_index:]

print("Content train:", len(content_train))
print("Content validation:", len(content_val))
print("Content test:", len(content_test))

Content train: 800
Content validation: 100
Content test: 100


In [23]:
#WikiArt 不能直接随机划分,WikiArt 图片位于不同风格文件夹